# PettingZoo agent — starter notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ml-arena/competition-baseline/blob/main/pettingzoo/agent_baseline.ipynb)

A minimal, runnable **multi-agent** baseline for any ml-arena PettingZoo competition
(Connect-Four, Chess, Texas Hold'em, the PettingZoo Atari suite, ...). It plays a
**random *legal* move** — it reads the `action_mask` and only picks allowed actions.

> Open in Colab, run top to bottom. Edit your API token and `COMPETITION_ID`
> (default `65` — **Connect Four**).

## 0. Setup

In [ ]:
!pip install -q "mlarena-sdk==0.3.0" numpy   # installs the `mlarena` package

import mlarena

API_TOKEN = "mlk_user_REPLACE_ME"   # <-- paste your token (Profile page -> API Keys)
COMPETITION_ID = 65

client = mlarena.connect(api_key=API_TOKEN, base_url="https://ml-arena.com")

## 1. Define your agent

The next cell writes **`agent.py`** to disk with `%%writefile`. We upload the whole file
(so its `import`s come along) — that is what the worker runs. Keep it **self-contained**:
it is executed on its own, so it may not import the environment or other notebook cells,
only public packages available in the runtime.

Same contract as single-agent, plus two multi-agent details:

- `reset(env_player_name, episode_index)` is called at the start of each episode — the
  executor may seat you as a different player each game; remember your role here.
- For turn-based board games the observation carries an **`action_mask`** (a 0/1 vector
  of legal actions). You **must** pick an action where the mask is `1`, or you forfeit.

In [ ]:
%%writefile agent.py
import random


class Agent:
    def __init__(self, observation_space, action_space):
        self.observation_space = observation_space
        self.action_space = action_space
        self.env_player_name = ''
        self.episode_index = 0

    def reset(self, env_player_name, episode_index):
        self.env_player_name = env_player_name
        self.episode_index = episode_index

    def choose_action(self, observation, reward=0.0, terminated=False,
                      truncated=False, info=None, action_mask=None):
        if terminated or truncated:
            return None
        if action_mask is not None:
            legal = [i for i, ok in enumerate(action_mask) if ok]
            if legal:
                # TODO: replace random choice with your policy over legal moves.
                return random.choice(legal)
        return self.action_space.sample()

## 2. (optional) Sanity-check it locally

Fake a Connect-Four action mask (7 columns, one full) and confirm we only play legal.

In [ ]:
import random
from agent import Agent

class _FakeDiscrete:
    def __init__(self, n): self.n = n
    def sample(self): return random.randrange(self.n)

agent = Agent(observation_space=None, action_space=_FakeDiscrete(7))
agent.reset('player_0', 0)
mask = [1, 1, 0, 1, 1, 1, 1]   # column 2 is full
print('legal action:', agent.choose_action(observation=None, action_mask=mask))

## 3. Submit

In [ ]:
# Uploads agent.py (with its imports intact), then creates + deploys the attachment.
result = client.submit(competition_id=COMPETITION_ID, files=["agent.py"])
print(result)

# Stream status until the run reaches a terminal state (deploy -> queue -> run -> scored).
for line in client.tail_logs(COMPETITION_ID, result["attache_agent_id"]):
    print(line)

## 4. Leaderboard

In [ ]:
client.leaderboard(COMPETITION_ID)

## 5. Where to go from here

- **Search / plan:** for perfect-information games (Connect-Four, Chess) a few plies of
  minimax or MCTS over the legal moves already crushes the random baseline.
- **Self-play RL:** train a policy/value net (AlphaZero-style) offline, upload the
  weights next to `agent.py`.
- Always filter by `action_mask` — an illegal move loses the game instantly.
- **Reuse this notebook** for any PettingZoo competition; only `COMPETITION_ID` changes.